# Train Explainable Stress Detection (Google Colab)

## Zip layout (what to upload)
**Easiest:** run `python colab_pack/build_colab_pack.py` from the project root, then zip the whole **`colab_pack/`** folder (includes notebook, light `frontend/` without `node_modules`, empty `model/saved/`).

**Or** build **`project.zip`** with only:
- `backend/` (without heavy `model/saved` weights)
- `frontend/` source only (`src/`, `public/`, `package.json`) — **not** `node_modules`
- `resources/`

**Omit:** `trained model colab/`, `node_modules/`, PDFs, and other large exports. Colab writes new weights under `backend/model/saved/` after training.

## Notebook steps
1. Unzip → locate `backend/requirements.txt`
2. Delete any accidentally included **`trained model colab`** folder under the project root
3. Install deps and run `backend/model/train.py` (Dreaddit + bundled hard-negative seed + top-level `resources/*.csv`)
4. Verify artifacts; optionally zip `model/saved` for download

**Runtime:** GPU recommended.

In [ ]:
# 1) Upload project.zip

from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))
ZIP_NAME = list(uploaded.keys())[0]
assert ZIP_NAME.lower().endswith(".zip"), "Upload a .zip file"

In [ ]:
# 2) Unzip, resolve paths, remove trained-model export folder if present

import glob
import os
import shutil
import zipfile

EXTRACT_DIR = "/content"

with zipfile.ZipFile(ZIP_NAME, "r") as zf:
    zf.extractall(EXTRACT_DIR)

candidates = glob.glob("/content/**/backend/requirements.txt", recursive=True)
if not candidates:
    raise RuntimeError(
        "Could not find backend/requirements.txt. Zip must contain backend/, frontend/, resources/."
    )

BACKEND_DIR = os.path.dirname(candidates[0])
PROJECT_DIR = os.path.dirname(BACKEND_DIR)
RESOURCES_DIR = os.path.join(PROJECT_DIR, "resources")


def _is_trained_model_colab_folder(name: str) -> bool:
    n = name.lower().replace("-", " ").replace("_", " ")
    return "trained" in n and "model" in n and "colab" in n


for entry in list(os.listdir(PROJECT_DIR)):
    full = os.path.join(PROJECT_DIR, entry)
    if os.path.isdir(full) and _is_trained_model_colab_folder(entry):
        print("Removing (not needed):", full)
        shutil.rmtree(full, ignore_errors=True)

print("PROJECT_DIR =", PROJECT_DIR)
print("BACKEND_DIR =", BACKEND_DIR)
print("RESOURCES_DIR exists:", os.path.isdir(RESOURCES_DIR))

In [ ]:
# 3) Install dependencies

import os
import subprocess
import sys

os.chdir(BACKEND_DIR)
print("cwd:", os.getcwd())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "accelerate", "safetensors", "tqdm"]
)
print("Dependencies ready.")

In [ ]:
# 4) (Optional) Inspect top-level CSVs in resources/

import glob
import os
import subprocess
import sys

os.chdir(BACKEND_DIR)

csv_paths = sorted(glob.glob(os.path.join(RESOURCES_DIR, "*.csv")))
print("resources/*.csv:", len(csv_paths))
for p in csv_paths:
    print(" -", p)

if csv_paths:
    cmd = [sys.executable, "model/dataset_inspect.py", "--csv_paths", *csv_paths]
    subprocess.check_call(cmd)
else:
    print("No CSVs: training uses Dreaddit + bundled hard-negative seed only.")

In [ ]:
# 5) Train

import glob
import os
import subprocess
import sys

os.chdir(BACKEND_DIR)

USE_DREADDIT = True
USE_MULTI_TASK = True
USE_HARD_NEGATIVE_SEED = True
EPOCHS = 4
BATCH_SIZE = 16
PRETRAINED_MODEL = "bert-base-uncased"

csv_paths = sorted(glob.glob(os.path.join(RESOURCES_DIR, "*.csv")))
cmd = [
    sys.executable,
    "model/train.py",
    "--epochs",
    str(EPOCHS),
    "--batch_size",
    str(BATCH_SIZE),
    "--pretrained_model",
    PRETRAINED_MODEL,
]
if csv_paths:
    cmd.append("--csv_paths")
    cmd.extend(csv_paths)
if not USE_DREADDIT:
    cmd.append("--no_dreaddit")
if USE_MULTI_TASK:
    cmd.append("--multi_task")
if not USE_HARD_NEGATIVE_SEED:
    cmd.append("--no_hard_negative_seed")

print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# 6) Verify artifacts in backend/model/saved

import os

save_dir = os.path.join(BACKEND_DIR, "model", "saved")
print("save_dir:", save_dir)

required = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "model.safetensors",
    "training_metadata.json",
    "threshold_analysis.json",
]
for f in required:
    p = os.path.join(save_dir, f)
    print(f, "OK" if os.path.isfile(p) else "MISSING")

aux = os.path.join(save_dir, "emotion_aux.joblib")
print(
    "emotion_aux.joblib",
    "OK" if os.path.isfile(aux) else "MISSING (expected if USE_MULTI_TASK was False)",
)

stm = os.path.join(save_dir, "special_tokens_map.json")
if not os.path.isfile(stm):
    print("special_tokens_map.json: OPTIONAL if tokenizer.json is present")

In [ ]:
# 7) (Optional) Download model/saved as a zip

import os
import shutil

from google.colab import files

save_dir = os.path.join(BACKEND_DIR, "model", "saved")
stage = "/content/stress_model_saved_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(save_dir, stage)
archive = shutil.make_archive(
    "/content/stress_model_saved",
    "zip",
    root_dir="/content",
    base_dir=os.path.basename(stage),
)
print("Archive:", archive)
files.download(archive)